> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, uploads the
> package and scans both I2C buses. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host,
>   writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

# Hardware and configuration

| | Host run | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C buses** | none (`bus_open` needs `machine`) | bus 0: STEMMA QT (scl=9 sda=8); bus 1: second Qwiic chain (scl=15 sda=16) |
| **Parts used** | none | whatever is wired: the scan only lists addresses |
| **Result** | not run yet | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): the simulation needs no board entry. `board_get` accepts a board dict,
and the simulated bench (`sim.bench`) passes the real board's dict so bus clocks are worked out
the same way as on the bench.

# Python file
location: `src/micropython_i2c/boards.py`

tests: `tests/test_boards.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/boards.py
"""Board pin maps, and ``bus_open`` for building a ``machine.I2C`` from one.

Add a board by adding an entry to ``BOARDS``. ``baud_max`` caps the bus clock below what the
parts allow, e.g. for long wires or weak pull-ups.

Generated by ``dev_notebooks/boards_devNB``: edit the notebook, not this file.
"""

# Imports
## External
`machine`, imported inside `bus_open` only, so the module imports on the host.

## Internal
None.

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_boards.py
import pytest

from micropython_i2c.boards import BOARDS, board_get, bus_open

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

# Intent
**Purpose:** one place that knows which pins each I2C bus uses on each board, and how fast
the board's wiring lets a bus run.

**Used by:** `I2CManager` (builds every bus of a routing table from the board entry and caps
the clock at `baud_max`), and every notebook's hardware section (`bus_open("feathers3", 0)`).

**Not its job:** which parts are on a bus (that is the routing table), or the clock the parts
allow (each driver's `BAUD_MAX`; the manager takes the lowest of them and the board's).

## What the module defines

| Name | What it is |
|---|---|
| `BOARDS` | `{board name: {"buses": {bus_id: {"scl", "sda", "baud_max"}}}}` |
| `board_get` | a board dict from a name or a dict |
| `bus_open` | a `machine.I2C` for one bus of a board (board only) |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/boards.py -a
__all__ = ["BOARDS", "board_get", "bus_open"]

## `BOARDS`
The FeatherS3's two Qwiic chains. Bus 0's pins are the STEMMA QT connector, with on-board
pull-ups. Bus 1's were confirmed on 2026-11-13, when the SparkFun mux answered there
(`device_notebooks/muxes/tca9548a_devNB`, "Find the mux").

In [ ]:
%%extract ../src/micropython_i2c/boards.py -a
#: Board name -> pin map. ``baud_max`` is what the board's wiring allows on that bus.
BOARDS = {
    # Unexpected Maker FeatherS3 (ESP32-S3). The STEMMA QT / Qwiic connector and the
    # SDA/SCL pads share GPIO 8/9 and have on-board pull-ups; bare header wiring does
    # not. The S3 routes I2C through the GPIO matrix, so peripheral 0 or 1 both work.
    "feathers3": {
        "buses": {
            0: {"scl": 9, "sda": 8, "baud_max": 400_000},
            # Second Qwiic chain. Confirmed 2026-11-13: the SparkFun mux answers here
            # (tca9548a_devNB, "Find the mux").
            1: {"scl": 15, "sda": 16, "baud_max": 400_000},
        },
    },
}

## `board_get`
Everything that takes a board accepts either its `BOARDS` name or the dict itself, so a test
or the simulation can pass a board that is not in `BOARDS`.

In [ ]:
%%extract ../src/micropython_i2c/boards.py -a
def board_get(board):
    """Return a board dict, given either a ``BOARDS`` key or the dict itself.

    Raises:
        ValueError: If ``board`` is a name not in ``BOARDS``.
    """
    if isinstance(board, dict):
        return board
    try:
        return BOARDS[board]
    except KeyError:
        raise ValueError("unknown board {!r}; known: {}".format(board, sorted(BOARDS))) from None

### validation/test

#### validation: nominal usage

In [ ]:
fs3 = board_get("feathers3")
print("feathers3 buses:", {k: (v["scl"], v["sda"], v["baud_max"]) for k, v in fs3["buses"].items()})
own = {"buses": {0: {"scl": 1, "sda": 2}}}
print("a dict passes through:", board_get(own) is own)

#### validation: anti-nominal usage

In [ ]:
try:
    board_get("pico")  # not in BOARDS
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("board_get('pico') should have raised ValueError")

#### unit

In [ ]:
%%extract ../tests/test_boards.py -a --strip-ipytest
%%ipytest


def test_board_get_by_name_and_by_dict():
    assert board_get("feathers3") is BOARDS["feathers3"]
    own = {"buses": {}}
    assert board_get(own) is own

In [ ]:
%%extract ../tests/test_boards.py -a --strip-ipytest
%%ipytest


def test_unknown_board_names_the_known_ones():
    with pytest.raises(ValueError, match="unknown board 'pico'; known: \\['feathers3'\\]"):
        board_get("pico")

In [ ]:
%%extract ../tests/test_boards.py -a --strip-ipytest
%%ipytest


def test_every_bus_has_pins_and_a_clock_cap():
    for name, board in BOARDS.items():
        for bus_id, cfg in board["buses"].items():
            assert {"scl", "sda", "baud_max"} <= set(cfg), (name, bus_id)
            assert 10_000 <= cfg["baud_max"] <= 1_000_000

## `bus_open`
A `machine.I2C` for one bus. `machine` is imported inside the function, so importing
`boards` on the host works; calling `bus_open` there raises `ImportError`.

In [ ]:
%%extract ../src/micropython_i2c/boards.py -a
def bus_open(board, bus_id=0, freq=100_000):
    """Create the ``machine.I2C`` for ``bus_id`` on ``board`` (board only).

    Raises:
        ImportError: On the host, which has no ``machine`` module.
        KeyError: If the board has no bus ``bus_id``.
    """
    from machine import I2C, Pin

    cfg = board_get(board)["buses"][bus_id]
    return I2C(bus_id, scl=Pin(cfg["scl"]), sda=Pin(cfg["sda"]), freq=freq)

### validation/test

#### validation: nominal usage
Validated on the board, in `# Hardware validation` below: the host has no `machine`.

#### validation: anti-nominal usage

In [ ]:
try:
    bus_open("feathers3", 0)  # the host has no machine module
except ImportError as err:
    print("raised ImportError, as expected:\n    {}".format(err))
else:
    raise AssertionError("bus_open on the host should have raised ImportError")

#### unit

In [ ]:
%%extract ../tests/test_boards.py -a --strip-ipytest
%%ipytest


def test_bus_open_needs_the_board():
    with pytest.raises(ImportError):
        bus_open("feathers3", 0)

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package and scans both I2C buses (a scan only sends address bytes).
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

Layout: connect and set up, the **matching tests**, then the tally and disconnect.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.boards import BOARDS, board_get, bus_open
from micropython_i2c.debug import check, check_raises, check_reset, check_summary, info
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)

check_reset()

## Matching tests

**Matches** `test_board_get_by_name_and_by_dict` and `test_unknown_board_names_the_known_ones`.

**What to do:** nothing.

In [ ]:
check("test_board_get_by_name_and_by_dict", board_get("feathers3") is BOARDS["feathers3"])
check_raises("test_unknown_board_names_the_known_ones", ValueError, board_get, "pico")

**Matches** `test_bus_open_needs_the_board`, the board's half: each bus opens on its pins and
scans. An empty scan is not a failure (nothing may be wired), but every address found is
printed.

**What to do:** nothing. Read the addresses: they should match README "The bench".

In [ ]:
for bus_id in sorted(BOARDS["feathers3"]["buses"]):
    try:
        found = bus_open("feathers3", bus_id).scan()
    except Exception as e:  # a pin problem raises rather than scanning empty
        check("bus {} opens".format(bus_id), False, repr(e))
        continue
    check("bus {} opens and scans".format(bus_id), True)
    info("bus {}: {}".format(bus_id, [hex(a) for a in found]))

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))